# HF365 x MODIS — Temporal Overlap Facts

Purely temporal facts about how HF365 and MODIS observation dates relate
to each other. No spatial content here -- see `03-control-points` for
real-world pixel locations. No correlation/modeling -- see
`02-correlation-analysis` for that.


In [ ]:
import os
import numpy as np
import pandas as pd

# vvv EDIT THESE to match your folders vvv
HF365_DIR = "."
MODIS_DATES_SOURCE = "modis_downloads"  # folder of MODIS .tif files (only dates are needed here)
# ^^^ EDIT THESE to match your folders vvv

HF365_FILES = {
    "EVI2": "hf365-01-hls-evi2.csv",
    "NDVI": "hf365-02-hls-ndvi.csv",
    "NIRv": "hf365-03-hls-nirv.csv",
}


## 1. Load both datasets' dates

In [ ]:
hf365_dates = set()
for fname in HF365_FILES.values():
    df = pd.read_csv(os.path.join(HF365_DIR, fname))
    date_col = df.columns[0]
    hf365_dates.update(pd.to_datetime(df[date_col]).tolist())
hf365_dates = sorted(hf365_dates)
print(f"HF365 distinct dates: {len(hf365_dates)}")


In [ ]:
import re
import glob
pattern = re.compile(r"MCD15A2H\.061_Lai_500m_(\d{8})T\d+_aid\d+")
files = glob.glob(os.path.join(MODIS_DATES_SOURCE, "**", "*.tif"), recursive=True)
modis_dates = sorted(set(
    pd.to_datetime(m.group(1), format="%Y%m%d")
    for f in files
    if (m := pattern.search(os.path.basename(f)))
))
print(f"MODIS distinct composite dates: {len(modis_dates)}")


## 2. Date ranges and overall overlap window

In [ ]:
hf365_min, hf365_max = hf365_dates[0], hf365_dates[-1]
modis_min, modis_max = modis_dates[0], modis_dates[-1]
overlap_start, overlap_end = max(hf365_min, modis_min), min(hf365_max, modis_max)

print(f"HF365 date range: {hf365_min.date()} to {hf365_max.date()}")
print(f"MODIS date range: {modis_min.date()} to {modis_max.date()}")
print(f"Overlapping window: {overlap_start.date()} to {overlap_end.date()} "
      f"({(overlap_end-overlap_start).days} days, {(overlap_end-overlap_start).days/365.25:.2f} years)")


## 3. HF365 observations per MODIS window

Each MODIS window's true length is the actual gap to the **next real
MODIS date** -- not an assumed fixed 8 days (real gaps are occasionally
5, 6, or 16 days).

In [ ]:
modis_in_range = [d for d in modis_dates if overlap_start <= d <= overlap_end]

window_records = []
for i in range(len(modis_in_range) - 1):
    w_start, w_end = modis_in_range[i], modis_in_range[i+1] - pd.Timedelta(days=1)
    gap_days = (modis_in_range[i+1] - w_start).days
    n_hf365 = sum(1 for d in hf365_dates if w_start <= d <= w_end)
    window_records.append(dict(modis_date=w_start, window_length_days=gap_days, hf365_obs_in_window=n_hf365))

window_df = pd.DataFrame(window_records)
print(f"MODIS windows evaluated: {len(window_df)}")
print("\nWindow length distribution:")
print(window_df["window_length_days"].value_counts().sort_index())
print("\nHF365 observations per window:")
print(window_df["hf365_obs_in_window"].value_counts().sort_index())
print(f"\nWindows with ZERO HF365 observations: "
      f"{(window_df['hf365_obs_in_window']==0).sum()} of {len(window_df)}")


## 4. Zero-match windows: how close is the nearest HF365 point?

In [ ]:
hf365_dates_arr = np.array(hf365_dates, dtype="datetime64[D]")
zero_windows = window_df[window_df["hf365_obs_in_window"] == 0].copy()

backward_gaps, forward_gaps = [], []
for _, row in zero_windows.iterrows():
    w_start = np.datetime64(row["modis_date"], "D")
    w_end = w_start + np.timedelta64(row["window_length_days"] - 1, "D")
    before = hf365_dates_arr[hf365_dates_arr < w_start]
    after = hf365_dates_arr[hf365_dates_arr > w_end]
    backward_gaps.append((w_start - before.max()).astype(int) if len(before) else np.nan)
    forward_gaps.append((after.min() - w_end).astype(int) if len(after) else np.nan)

zero_windows["backward_gap_to_prev_hf365"] = backward_gaps
zero_windows["forward_gap_to_next_hf365"] = forward_gaps

print(f"Zero-match windows analyzed: {len(zero_windows)}")
print("\nBackward-gap summary (days to the most recent PRIOR HF365 date):")
print(zero_windows["backward_gap_to_prev_hf365"].describe())


## 5. Summary

In [ ]:
print("=== TEMPORAL OVERLAP FACT SHEET ===")
print(f"HF365 date range: {hf365_min.date()} to {hf365_max.date()}")
print(f"MODIS date range: {modis_min.date()} to {modis_max.date()}")
print(f"Overlap window: {overlap_start.date()} to {overlap_end.date()}")
print(f"MODIS windows evaluated: {len(window_df)}")
print(f"MODIS windows with zero HF365 observations: {(window_df['hf365_obs_in_window']==0).sum()} "
      f"({100*(window_df['hf365_obs_in_window']==0).mean():.1f}%)")
